In [1]:
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.ui.workspace import CloudWorkspace

## Configuration credential

In [2]:
# Configuration for the credential
import os
from dotenv import load_dotenv

ENV_PATH = ".env"
load_dotenv(ENV_PATH)

False

## Create project on evidently AI

In [3]:
ws = CloudWorkspace(token=os.getenv("EVIDENTLY_TOKEN"), url="https://app.evidently.cloud")

## Create a Project within your Organization, or connect to an existing Project:

In [4]:
try:
    existing_projects = ws.list_projects()
    project = next((p for p in existing_projects if p.name == "LLM Audit"), None)
    
    if project is None:
        project = ws.create_project(name="LLM Audit", org_id="019d6885-06c6-7601-9a3e-2298bed991f4")
        project.description = "Project for auditing LLM performance on models, data, and LLMs evaluations."
        project.save()
        print(f"✅ Project created: {project.name}")   # ← print, not project()
    else:
        print(f"📁 Project already exists: {project.name}")  # ← print, not project()

except Exception as e:
    print(f"Error accessing or creating project: {e}")

print(f"Project name: {project.name}")

📁 Project already exists: LLM Audit
Project name: LLM Audit


## Prepare the dataset -> Merge prompt with dataframe into reliable data information
* ### Load dataset
* ### Create prompt

## Load dataset

In [5]:
pd.set_option('display.max_columns', None)
df = pd.read_parquet("../../Database/data/eda_banking.parquet")
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,11983969.0,1,1,1,10134888.0,1,1,2,DIAMOND,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,8380786.0,1,0,1,11254258.0,0,1,3,DIAMOND,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,15966080.0,3,1,0,11393157.0,1,1,3,DIAMOND,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,11983969.0,2,0,0,9382663.0,0,0,5,GOLD,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,12551082.0,1,1,1,7908410.0,0,0,5,GOLD,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,11983969.0,2,1,0,9627064.0,0,0,1,DIAMOND,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,5736961.0,1,1,1,10169977.0,0,0,5,PLATINUM,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,11983969.0,1,0,1,4208558.0,1,1,3,SILVER,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,7507531.0,2,1,0,9288852.0,1,1,2,GOLD,339,1,2,25025.103333,0,0,0,1,1,1,3


## Create LLMs prompt for evidentlyAI production

In [6]:
# Define questions and context
question_and_context = [
    [
        "What is the fraud detection rate in the dataset?",
        "Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical."
    ],
    [
        "Which ML model performs best for credit default prediction?",
        "XGBoost and Random Forest outperform logistic regression on imbalanced datasets."
    ],
    [
        "How should missing values be handled in operational banking data?",
        "Imputation must preserve regulatory compliance and not introduce synthetic bias."
    ],
    [
        "What is the churn rate and how can it be reduced?",
        "Customer churn in retail banking averages 15-20% annually."
    ],
    [
        "Explain the class imbalance issue in fraud detection.",
        "Fraud datasets are heavily imbalanced with fraud rates below 1-2%."
    ],
]

pd.set_option('display.max_colwidth', None)
eval_df = pd.DataFrame(question_and_context, columns=["question", "context"])

In [7]:
eval_df.head()

,question,context
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.
3,What is the churn rate and how can it be reduced?,Customer churn in retail banking averages 15-20% annually.
4,Explain the class imbalance issue in fraud detection.,Fraud datasets are heavily imbalanced with fraud rates below 1-2%.


In [8]:
from groq import Groq
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

client = Groq(api_key=os.environ.get("GROQ_API_KEY_2"))

# 1. Build data context from dataframe
def build_data_context(df):
    context = f"""
You are a banking data analyst & data scentist. Use the following dataset information to reasong and answer questions.

DATASET OVERVIEW:
- Total Records: {len(df):,}
- Columns: {list(df.columns)}

STATISTICAL SUMMARY:
{df.describe().to_string()}

CATEGORICAL COLUMNS:
- Geography Distribution: {df['Geography'].value_counts().to_dict()}
- Gender Distribution: {df['Gender'].value_counts().to_dict()}
- Card Type Distribution: {df['Card Type'].value_counts().to_dict()}

- KEY METRICS:
- Fraud Rate: {df['Fraud'].mean():.2%}
- Churn Rate (Exited): {df['Exited'].mean():.2%}
- Avg Credit Score: {df['CreditScore'].mean():.2f}
- Avg Balance : {df['Balance'].mean():.2f}
- Avg Risk Score: {df['RiskScore'].mean():.2f}
- High Value Customers: {df['HighValueCustomer'].sum():,} ({df['HighValueCustomer'].mean():.2%})
- Customers with Complaints: {df['Complain'].sum():,} ({df['Complain'].mean():.2%})
- Active Members: {df['IsActiveMember'].sum():,} ({df['IsActiveMember'].mean():.2%})

CORRELATION WITH FRAUD (top features):
{df.corr(numeric_only=True)['Fraud'].sort_values(ascending=False).to_string()}

CORRELATION WITH CHURN (top features):
{df.corr(numeric_only=True)['Exited'].sort_values(ascending=False).to_string()}
"""
    
    return context

data_context = build_data_context(df)


In [9]:

# 2. Generate answers with data-aware prompting
def get_groq_response_with_context(row):
    prompt = f"""
{data_context}

---
CONTEXT HINT: {row['context']}
QUESTION: {row['question']}

Answer based strictly on the dataset statistics provided above. Be specific with numbers and percentages from the data.
"""
    
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1024,
        top_p=0.9,
        stream=False
    )
    return completion.choices[0].message.content

In [10]:
# apply to eval_df
eval_df['answer'] = eval_df.apply(get_groq_response_with_context, axis=1)
eval_df

,question,context,answer
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"To determine the fraud detection rate in the dataset, we need to understand the given statistics. The fraud rate is directly provided in the key metrics section.\n\n- Fraud Rate: 3.32%\n\nThis means that out of the total records (10,000), 3.32% of the customers are involved in fraudulent activities. To find the exact number of fraudulent cases, we can calculate:\n\nFraudulent Cases = Total Records * Fraud Rate\n= 10,000 * 3.32%\n= 10,000 * 0.0332\n= 332\n\nTherefore, there are 332 cases of fraud in the dataset, which accounts for 3.32% of the total records."
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"Based on the dataset statistics provided, the best ML model for credit default prediction (Fraud) is likely to be either XGBoost or Random Forest. \n\nThe key reasons for this are:\n1. **Imbalanced dataset**: The Fraud Rate is 3.32%, indicating an imbalanced dataset where the minority class (fraudulent cases) is significantly smaller than the majority class (non-fraudulent cases). \n2. **Model performance on imbalanced datasets**: As hinted in the context, XGBoost and Random Forest tend to outperform logistic regression on imbalanced datasets. \n3. **Correlation with Fraud**: The top features correlated with Fraud include Exited (0.292838), ComplainFlag (0.292243), Complain (0.292243), OperationalRiskScore (0.283743), and RiskScore (0.223436). These features can be effectively utilized by XGBoost and Random Forest to predict credit default.\n\nWhile the exact performance of XGBoost and Random Forest is not provided in the dataset statistics, their ability to handle imbalanced datasets and complex interactions between features makes them well-suited for credit default prediction.\n\nIn contrast, logistic regression may not perform as well due to its limitations in handling imbalanced datasets and non-linear relationships between features. \n\nTherefore, based on the provided dataset statistics, it is reasonable to conclude that either XGBoost or Random Forest would be a better choice for credit default prediction, with an expected improvement in performance compared to logistic regression. However, the exact performance difference would depend on the specific implementation and hyperparameter tuning of the models."
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.,"Based on the provided dataset statistics, handling missing values in operational banking data requires careful consideration to preserve regulatory compliance and avoid introducing synthetic bias. Here are some insights from the data:\n\n1. **Missing values are not explicitly mentioned**: The statistical summary does not indicate the presence of missing values. However, this does not necessarily mean that there are no missing values in the dataset.\n2. **Data distribution and correlations**: The dataset provides a comprehensive overview of the distribution of various features, including means, standard deviations, and correlations. These statistics can be used to inform imputation strategies.\n3. **Fraud and churn correlations**: The correlations with fraud and churn (Exited) provide valuable insights into the relationships between features. For example, the top features correlated with fraud are Exited (0.292838), ComplainFlag (0.292243), and Complain (0.292243). Similarly, the top features correlated with churn are ComplainFlag (0.995693), Complain (0.995693), and RiskScore (0.589059).\n\nTo handle missing values, I would recommend the following approaches:\n\n1. **Mean/Median imputation**: For features with a relatively low number of missing values (e.g., less than 1%), mean or median imputation can be used. 

## Auditing the LLMs for evaluation

* ###  Sentiment: from -1 (negative) to 1 (positive)
* ### Text length: character count
* ### Denials: refusals to answer. This uses an LLM-as-a-judge with built-in prompt.

### Each evaluation is a descriptor. It adds a new score or label to each row in your dataset. For LLM-as-a-judge, we’ll use OpenAI GPT-4o mini. Set OpenAI key as an environment variable:

In [12]:
import litellm
import warnings
import time
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY_2", "")
print(f"Key loaded: {bool(os.environ.get('GROQ_API_KEY_2'))}")

# ── Option 1: Tell LiteLLM to auto-retry with backoff ─────────────────────
litellm.num_retries = 5
litellm.retry_after = 10  # wait 10s between retries

eval_dataset = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        Sentiment("answer", alias="Sentiment"),
        TextLength("answer", alias="Length"),
        DeclineLLMEval(
            "answer",
            alias="Denials",
            provider="litellm",
            model="groq/llama-3.3-70b-versatile"
        ),
    ]
)

eval_dataset.as_dataframe()

Key loaded: True


,question,context,answer,Sentiment,Length,Denials,Denials reasoning
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"To determine the fraud detection rate in the dataset, we need to understand the given statistics. The fraud rate is directly provided in the key metrics section.\n\n- Fraud Rate: 3.32%\n\nThis means that out of the total records (10,000), 3.32% of the customers are involved in fraudulent activities. To find the exact number of fraudulent cases, we can calculate:\n\nFraudulent Cases = Total Records * Fraud Rate\n= 10,000 * 3.32%\n= 10,000 * 0.0332\n= 332\n\nTherefore, there are 332 cases of fraud in the dataset, which accounts for 3.32% of the total records.",-0.9823,552,OK,"The text provided does not contain a refusal or a polite rejection to do something, but rather an explanation and calculation of the fraud detection rate in a dataset, indicating that the text is not a decline but rather a neutral or informative passage."
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"Based on the dataset statistics provided, the best ML model for credit default prediction (Fraud) is likely to be either XGBoost or Random Forest. \n\nThe key reasons for this are:\n1. **Imbalanced dataset**: The Fraud Rate is 3.32%, indicating an imbalanced dataset where the minority class (fraudulent cases) is significantly smaller than the majority class (non-fraudulent cases). \n2. **Model performance on imbalanced datasets**: As hinted in the context, XGBoost and Random Forest tend to outperform logistic regression on imbalanced datasets. \n3. **Correlation with Fraud**: The top features correlated with Fraud include Exited (0.292838), ComplainFlag (0.292243), Complain (0.292243), OperationalRiskScore (0.283743), and RiskScore (0.223436). These features can be effectively utilized by XGBoost and Random Forest to predict credit default.\n\nWhile the exact performance of XGBoost and Random Forest is not provided in the dataset statistics, their ability to handle imbalanced datasets and complex interactions between features makes them well-suited for credit default prediction.\n\nIn contrast, logistic regression may not perform as well due to its limitations in handling imbalanced datasets and non-linear relationships between features. \n\nTherefore, based on the provided dataset statistics, it is reasonable to conclude that either XGBoost or Random Forest would be a better choice for credit default prediction, with an expected improvement in performance compared to logistic regression. However, the exact performance difference would depend on the specific implementation and hyperparameter tuning of the models.",0.9272,1629,OK,"The text does not contain any refusal or rejection, instead it provides an analysis and recommendation for the best ML model for credit default prediction, indicating a positive and helpful response."
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.,"Based on the provided dataset statistics, handling missing values in operational banking data requires careful consideration to preserve regulatory compliance and avoid introducing synthetic bias. Here are some insights from the data:\n\n1. **Missing values are not explicitly mentioned**: The statistical summary does not indicate the presence of missing values. However, this does not necessarily mean that there are no missing values in the dataset.\n2. **Data distribution and correlations**: The dataset provides a comprehensive overview of the distribution of various features, including means, standard deviations, and correlations. These statistics can be used to inform imputation strategies.\n3. **Fraud and churn correlations**: The correlations with fraud and churn (Exited) provide valuab

## Deploy to evidentlyAI UI Dashboard

In [56]:
from evidently.ui.workspace import DashboardPanelPlot, DashboardModel, DashboardTabModel
from evidently.sdk.models import PanelMetric

# 1. Define panels with correct PanelMetric fields
panel_sentiment = DashboardPanelPlot(
    title="Answer Sentiment",
    size="full",
    values=[
        PanelMetric(
            metric="Sentiment",
            legend="Sentiment Score",
        )
    ],
    plot_params={"plot_type": "line"}
)

panel_length = DashboardPanelPlot(
    title="Answer Text Length",
    size="full",
    values=[
        PanelMetric(
            metric="Denials",
            legend="Denial Rate"
        )
    ],
    plot_params={"plot_type": "bar"}
)

panel_denials = DashboardPanelPlot(
    title="LLM Denial Rate",
    size="full",
    values=[
        PanelMetric(
            metric="Denials",
            legend="Denial Rate"
        )
    ],
    plot_params={"plot_type": "bar"}
)

# 2. Create tab grouping all panels
tab = DashboardTabModel(
    title="LLM Audit Overview",
    panels=[panel_sentiment.id, panel_length.id, panel_denials.id]
)

# 3. Build dashboard model
dashboard = DashboardModel(
    tabs=[tab],
    panels=[panel_sentiment, panel_length, panel_denials]
)

# 4. Push to Evidently Cloud
ws.save_dashboard(project.id, dashboard)
print("✅ Dashboard saved to Evidently Cloud")
print(f"🔗 https://app.evidently.cloud/projects/{project.id}/dashboard")

✅ Dashboard saved to Evidently Cloud
🔗 https://app.evidently.cloud/projects/019d7204-9bfc-76e9-8c54-ee541e230e6f/dashboard
